# ResaleRadar - Exploratory Data Analysis

Dataset: *Used Phones & Tablets Pricing Dataset* (ahsan81, Kaggle).  
Target: `normalized_used_price`.

> Run `python src/train_model.py` first so that the Random Forest feature-importance cell (section 7) can load the trained model. If the model is missing, the notebook trains a quick one.

In [ ]:
# Imports and project paths
import sys
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

# Works whether the notebook is started from notebooks/ or from the project root.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
DATA_PATH = ROOT / "data" / "used_device_data.csv"
TARGET = "normalized_used_price"
print("Project root:", ROOT)

## 1. Load dataset: shape, dtypes, head

In [ ]:
df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
print()
print(df.dtypes)
df.head()

## 2. Missing value heatmap

In [ ]:
plt.figure(figsize=(12, 5))
sns.heatmap(df.isnull(), cbar=False, yticklabels=False, cmap="viridis")
plt.title("Missing values (yellow = missing)")
plt.show()

missing = df.isnull().sum()
print(missing[missing > 0] if (missing > 0).any() else "No missing values.")

## 3. Distribution of `normalized_used_price`

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(df[TARGET], kde=True, bins=40, color="steelblue")
plt.title("Distribution of normalized used price")
plt.xlabel(TARGET)
plt.show()
print(df[TARGET].describe())

## 4. Correlation heatmap (numeric features)

In [ ]:
numeric_df = df.select_dtypes(include=[np.number])
plt.figure(figsize=(12, 9))
sns.heatmap(numeric_df.corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0, square=True)
plt.title("Correlation of numeric features")
plt.show()

## 5. Boxplots: price vs OS and price vs 4G

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.boxplot(data=df, x="os", y=TARGET, ax=axes[0])
axes[0].set_title("Price vs operating system")
sns.boxplot(data=df, x="4g", y=TARGET, ax=axes[1])
axes[1].set_title("Price vs 4G availability")
plt.tight_layout()
plt.show()

## 6. Scatter plots: RAM vs price, Battery vs price

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.scatterplot(data=df, x="ram", y=TARGET, alpha=0.5, ax=axes[0])
axes[0].set_title("RAM (GB) vs price")
sns.scatterplot(data=df, x="battery", y=TARGET, alpha=0.5, ax=axes[1], color="darkorange")
axes[1].set_title("Battery (mAh) vs price")
plt.tight_layout()
plt.show()

## 7. Feature importance (Random Forest)

In [ ]:
from preprocess import FEATURE_COLUMNS, clean_data, fit_preprocessor

rf_path = ROOT / "models" / "random_forest.pkl"
if rf_path.exists():
    rf = joblib.load(rf_path)          # model trained by src/train_model.py
    print("Loaded trained model:", rf_path.name)
else:
    from sklearn.ensemble import RandomForestRegressor
    print("Trained model not found - training a quick Random Forest for this plot...")
    X, y, _ = fit_preprocessor(clean_data(df), save=False)
    rf = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1).fit(X, y)

importances = pd.Series(rf.feature_importances_, index=FEATURE_COLUMNS).sort_values()
plt.figure(figsize=(9, 6))
importances.plot.barh(color="seagreen")
plt.title("Random Forest feature importance")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()
importances.sort_values(ascending=False).round(4)